# Lab 10 — Part A: Lakehouse Federation

**Goal:** Query Neon PostgreSQL through a Unity Catalog foreign catalog and join external data with a local Delta table.

**External source:** Neon PostgreSQL

**Foreign catalog:** neon_postgres_conn_catalog

**Local Delta schema:** workspace.lab_10_federation_cdc

This notebook demonstrates:
1. Querying external PostgreSQL data without copying it.
2. Joining external PostgreSQL orders with local Delta customers.
3. Understanding Federation versus Delta ingestion.

All operations are read-only.

In [0]:
# WHAT: Read data directly from Neon PostgreSQL.
# WHY: Confirm that Lakehouse Federation is working.

customers = spark.sql("""
    SELECT COUNT(*) AS total
    FROM neon_postgres_conn_catalog.public.lab10_customers
""").first()["total"]

orders = spark.sql("""
    SELECT COUNT(*) AS total
    FROM neon_postgres_conn_catalog.public.lab10_orders
""").first()["total"]

print(f"Neon customers: {customers}")
print(f"Neon orders: {orders}")

assert customers == 12, "Unexpected customer count"
assert orders == 20, "Unexpected order count"

print("PASS: Lakehouse Federation")

In [0]:
# WHAT: Join external PostgreSQL orders with local Delta customers.
# WHY: Demonstrate a hybrid Lakehouse Federation query.

joined = spark.sql("""
    SELECT
        o.order_id,
        o.customer_id,
        c.first_name,
        c.city,
        o.amount,
        o.status,
        CASE
            WHEN c.customer_id IS NULL THEN 'NO_DELTA_MATCH'
            ELSE 'MATCHED'
        END AS join_status
    FROM neon_postgres_conn_catalog.public.lab10_orders o
    LEFT JOIN workspace.lab_10_federation_cdc.customers_delta c
        ON o.customer_id = c.customer_id
    ORDER BY o.order_id
""")

display(joined)

# A LEFT JOIN must retain every external order.
assert joined.count() == orders, "Order count mismatch"

# Every order_id should appear exactly once.
assert joined.select("order_id").distinct().count() == orders

display(joined.groupBy("join_status").count())

print("PASS: Federation + Delta JOIN")

In [0]:
# WHAT: Inspect federated query planning.
# WHY: Understand how Databricks accesses external data.

federation_plan = spark.sql("""
    EXPLAIN EXTENDED
    SELECT customer_id, amount
    FROM neon_postgres_conn_catalog.public.lab10_orders
    WHERE amount > 100
""")

display(federation_plan)

# Compare with a local Delta query.
delta_plan = spark.sql("""
    EXPLAIN EXTENDED
    SELECT customer_id, city
    FROM workspace.lab_10_federation_cdc.customers_delta
    WHERE city = 'Krakow'
""")

display(delta_plan)